# Empirical Orthogonal Function (EOF) analysis

Empirical Orthogonal Functions (EOFs) decompose a space–time field into a set of fixed spatial patterns (the EOFs) multiplied by time series (the principal components, PCs):

$$ T'(x, y, t) = \sum_{k} \mathrm{PC}_k(t) \, \mathrm{EOF}_k(x, y) .$$

The EOFs are orthogonal in space and the PCs are uncorrelated in time. They are ordered so that the first mode explains the largest fraction of the variance of $T'$, the second mode explains the largest fraction of what is left, and so on. EOF analysis is the same as principal component analysis (PCA) applied to a geophysical field; the EOFs are the eigenvectors of the spatial covariance matrix of $T'$.

This recipe uses the [`eofs`](https://ajdawson.github.io/eofs/latest/) package to do an EOF analysis of monthly sea surface temperature (SST) anomalies in the tropical Pacific from the ACCESS-OM2 0.25° interannually forced (IAF) run, 1958–2018. The first EOF is the El Niño–Southern Oscillation (ENSO) pattern. We check this by comparing its principal component with the Niño 3.4 index computed from the same model output.

The steps are:
1. Load monthly SST and grid-cell area.
2. Remove the seasonal cycle and the linear trend to obtain SST anomalies.
3. Weight each grid cell by the square root of its area.
4. Compute the EOFs, PCs and the fraction of variance explained by each mode.
5. Compare the leading PC with the Niño 3.4 index.

#### Converting to MOM6

This notebook uses MOM5 output (ACCESS-OM2). To use MOM6 output (e.g. ACCESS-OM3) change the following names:

| MOM5 | MOM6 |
|---|---|
| `xt_ocean` | `xh` |
| `yt_ocean` | `yh` |
| `sst` | `tos` |
| `area_t` | `areacello` |

The EOF analysis itself does not depend on the model.

In [ ]:
import intake
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cft
import cmocean.cm as cmo
from eofs.xarray import Eof

from dask.distributed import Client

In [ ]:
client = Client(threads_per_worker=1)
client

## Load the data

We use the ACCESS-NRI intake catalog and the 0.25° IAF experiment, which is forced with the JRA55-do atmospheric reanalysis from 1958 to 2018. Because the forcing is the observed atmosphere, the model's ENSO events should line up in time with the observed ones.

In [ ]:
catalog = intake.cat.access_nri

experiment = '025deg_jra55_iaf_omip2_cycle6'

start_time, end_time = '1958-01-01', '2018-12-31'

Select the tropical Pacific, 20°S–20°N and 120°E–80°W. ACCESS-OM2 longitudes run from −280° to 80°, so 120°E–80°W is −240° to −80°. After selecting the region we convert longitudes to the 0°–360° range (120° to 280°), which is easier to read and plot across the dateline.

In [ ]:
lat_range = slice(-20, 20)
lon_range = slice(-240, -80)

def to_0_360(da):
    return da.assign_coords(xt_ocean=da.xt_ocean % 360)

In [ ]:
sst = catalog[experiment].search(
    variable='sst', frequency='1mon', variable_cell_methods='time: mean'
).to_dask(
    xarray_open_kwargs=dict(use_cftime=True, chunks={}),
    xarray_combine_by_coords_kwargs=dict(compat='override', data_vars='minimal', coords='minimal'),
)['sst']

sst = sst.sel(time=slice(start_time, end_time), yt_ocean=lat_range, xt_ocean=lon_range)
sst = to_0_360(sst)
sst

The SST output is in Kelvin; convert it to °C. (An anomaly is the same in both units, but °C is more familiar on plots.) Then load the data into memory: the tropical Pacific at 0.25° for 61 years is only a few hundred MB.

In [ ]:
if sst.attrs.get('units', '').strip() in ('K', 'kelvin', 'Kelvin', 'degrees K'):
    sst = sst - 273.15
sst.attrs['units'] = '°C'

sst = sst.load()

In [ ]:
# the grid doesn't change, so we only need one of the grid files
area_search = catalog[experiment].search(variable='area_t', frequency='fx')
area = area_search.search(path=area_search.df.path[0]).to_dask()['area_t']
area = to_0_360(area.sel(yt_ocean=lat_range, xt_ocean=lon_range)).load()

# land cells have no SST; give them zero area so they never contribute
area = area.where(sst.isel(time=0).notnull(), 0)

## SST anomalies

EOFs describe variability about the mean, so we first remove

1. the **seasonal cycle**, i.e. the mean for each calendar month; otherwise the first EOF would just be the annual cycle, and
2. the **linear trend** at each grid point; otherwise the warming over 1958–2018 would appear as one of the leading modes.

The trend is fitted by least squares against time in months.

In [ ]:
climatology = sst.groupby('time.month').mean('time')
sst_anom = sst.groupby('time.month') - climatology

t = xr.DataArray(np.arange(sst_anom.sizes['time'], dtype=float), dims='time', coords={'time': sst_anom.time})
t = t - t.mean()
slope = (sst_anom * t).sum('time') / (t**2).sum()   # °C per month

sst_anom = sst_anom - sst_anom.mean('time') - slope * t
sst_anom = sst_anom.drop_vars('month')
sst_anom.attrs['units'] = '°C'

Here is the linear trend we removed, in °C per decade:

In [ ]:
fig = plt.figure(figsize=(10, 3.5))
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=180))
(slope * 120).where(area > 0).plot(
    ax=ax, transform=ccrs.PlateCarree(), cmap=cmo.balance, vmin=-0.4, vmax=0.4,
    cbar_kwargs={'label': 'SST trend (°C per decade)', 'shrink': 0.8},
)
ax.add_feature(cft.LAND, color='0.8')
ax.coastlines(linewidth=0.5)
ax.set_title('Linear SST trend, 1958–2018 (removed before the EOF analysis)');

## Weights

Grid cells have different areas, and a plain EOF analysis treats every grid point equally. To give each region an influence proportional to its area, we weight each cell by $\sqrt{\text{area}}$. The square root is used because EOFs come from the covariance matrix, which is a product of two anomaly fields, so the variance ends up weighted by area.

On a regular latitude–longitude grid people often use $\sqrt{\cos(\text{latitude})}$; using the model's own cell area does the same thing and also works on curvilinear grids.

In [ ]:
weights = np.sqrt(area / area.mean()).values  # eofs expects a plain numpy array

## Compute the EOFs

Create an `Eof` solver. It does the decomposition once, and then we can ask it for EOFs, PCs and variance fractions. The time dimension must be the first dimension and be called `time`. Land points (`NaN`) are fine as long as they are missing at every time.

In [ ]:
solver = Eof(sst_anom, weights=weights)

How much variance does each mode explain? The error bars are the typical sampling errors from [North et al. (1982)](https://doi.org/10.1175/1520-0493(1982)110%3C0699:SEITEO%3E2.0.CO;2). If the error bars of two neighbouring modes overlap, those modes are not well separated and can mix with each other.

In [ ]:
n_modes = 10
variance_fraction = solver.varianceFraction(neigs=n_modes) * 100
north_errors = solver.northTest(neigs=n_modes, vfscaled=True) * 100

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(np.arange(1, n_modes + 1), variance_fraction, yerr=north_errors, fmt='o-', capsize=3)
ax.set_xlabel('EOF mode')
ax.set_ylabel('variance explained (%)')
ax.set_xticks(np.arange(1, n_modes + 1))
ax.grid(alpha=0.3)
ax.set_title('Fraction of SST anomaly variance explained');

for k in range(3):
    print(f'EOF{k+1}: {variance_fraction[k].item():.1f} ± {north_errors[k].item():.1f} %')

Now get the first three EOFs and PCs. With `pcscaling=1` the PCs have unit variance, and `eofsAsCovariance` then gives each EOF in the units of the data (°C): it is the SST anomaly that goes with a one-standard-deviation value of the PC. These patterns are easier to interpret than the dimensionless eigenvectors.

In [ ]:
n = 3
eofs = solver.eofsAsCovariance(neofs=n, pcscaling=1)
pcs = solver.pcs(npcs=n, pcscaling=1)

## Is EOF1 ENSO? Compare with the Niño 3.4 index

The Niño 3.4 index is the area-averaged SST anomaly over 5°S–5°N, 170°W–120°W (190°–240° in 0°–360° longitude), smoothed with a 3-month running mean.

In [ ]:
nino34_region = dict(yt_ocean=slice(-5, 5), xt_ocean=slice(190, 240))

nino34 = sst_anom.sel(**nino34_region).weighted(area.sel(**nino34_region)).mean(['yt_ocean', 'xt_ocean'])
nino34 = nino34.rolling(time=3, center=True).mean()

The sign of an EOF and its PC is arbitrary: flipping both gives the same product. We choose the sign so that a positive PC1 means El Niño (positive correlation with Niño 3.4), and so that EOF2 is positive in the eastern equatorial Pacific (east of 240°).

In [ ]:
if xr.corr(pcs.sel(mode=0), nino34) < 0:
    eofs[0] = -eofs[0]
    pcs[:, 0] = -pcs[:, 0]

if eofs.sel(mode=1, yt_ocean=slice(-5, 5), xt_ocean=slice(240, 280)).mean() < 0:
    eofs[1] = -eofs[1]
    pcs[:, 1] = -pcs[:, 1]

r = xr.corr(pcs.sel(mode=0), nino34).item()
print(f'Correlation between PC1 and the Niño 3.4 index: {r:.2f}')

## Plot the EOF patterns

In [ ]:
fig, axes = plt.subplots(n, 1, figsize=(10, 9), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=180)})

for k, ax in enumerate(axes):
    p = eofs.sel(mode=k).plot(
        ax=ax, transform=ccrs.PlateCarree(), cmap=cmo.balance, vmin=-1, vmax=1,
        add_colorbar=False,
    )
    ax.add_feature(cft.LAND, color='0.8')
    ax.coastlines(linewidth=0.5)
    ax.set_title(f'EOF{k+1} ({variance_fraction[k].item():.1f}% of variance)')

# outline the Niño 3.4 region on EOF1
axes[0].plot([190, 240, 240, 190, 190], [-5, -5, 5, 5, -5], 'k--', linewidth=1, transform=ccrs.PlateCarree())

fig.colorbar(p, ax=axes, shrink=0.6, label='SST anomaly per 1 std of PC (°C)');

- **EOF1** is the ENSO pattern: warming along the equatorial cold tongue, strongest in the central and eastern Pacific, with weak cooling in the far western Pacific and the subtropics. It explains about half of the variance, far more than any other mode.
- **EOF2** has the opposite sign in the far eastern Pacific (warm) and the central Pacific (cool). Its time series below shows what it does.
- **EOF3** is a narrow equatorial band; it adds detail to how the cold tongue changes during and after events.

## Plot the principal components

We plot against decimal year and shade three strong El Niño events (1982–83, 1997–98 and 2015–16).

In [ ]:
decimal_year = pcs.time.dt.year + (pcs.time.dt.month - 0.5) / 12

fig, axes = plt.subplots(n, 1, figsize=(10, 7), sharex=True)

for k, ax in enumerate(axes):
    ax.plot(decimal_year, pcs.sel(mode=k), color='k', linewidth=1, label=f'PC{k+1}')
    ax.axhline(0, color='0.5', linewidth=0.5)
    ax.set_ylabel(f'PC{k+1}')
    for start, end in [(1982.5, 1983.5), (1997.4, 1998.5), (2015.3, 2016.4)]:
        ax.axvspan(start, end, color='tab:red', alpha=0.15)

nino34_std = (nino34 - nino34.mean()) / nino34.std()
axes[0].plot(decimal_year, nino34_std, color='tab:red', linewidth=1, label='Niño 3.4 (standardised)')
axes[0].legend(loc='upper left', ncol=2)
axes[0].set_title(f'PC1 vs Niño 3.4: correlation {r:.2f}')
axes[-1].set_xlabel('year')
axes[-1].set_xlim(decimal_year[0], decimal_year[-1]);

PC1 follows the Niño 3.4 index almost exactly, so EOF1 is ENSO. Positive peaks are El Niño events and negative ones La Niña.

PC2 has large positive peaks only during the two strongest El Niños, 1982–83 and 1997–98, when the warming reached all the way to the South American coast. During the 2015–16 El Niño, whose warming was centred further west, PC2 stays near zero. A single pattern (EOF1) cannot describe both kinds of event, so EOF2 corrects it: adding a positive EOF2 to EOF1 moves the warming east.

## A nonlinear relation between PC1 and PC2

PCs are uncorrelated by construction, but they need not be independent. A scatter plot of PC2 against PC1 shows a U-shape: PC2 is large when PC1 is very large (extreme El Niño) and slightly positive for strong La Niña, while PC2 is negative for moderate warm and cold conditions. This is the asymmetry between El Niño and La Niña, and between extreme and moderate El Niños, discussed by e.g. [Takahashi et al. (2011)](https://doi.org/10.1029/2011GL047364) and [Dommenget et al. (2013)](https://doi.org/10.1007/s00382-012-1475-0).

We fit a parabola to show this.

In [ ]:
pc1, pc2 = pcs.sel(mode=0), pcs.sel(mode=1)
decimal_year = pc1.time.dt.year + (pc1.time.dt.month - 0.5) / 12

coeffs = np.polyfit(pc1, pc2, deg=2)
x = np.linspace(pc1.min(), pc1.max(), 100)

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(pc1, pc2, c=decimal_year, s=6, cmap='viridis')
ax.plot(x, np.polyval(coeffs, x), 'k', linewidth=2,
        label=f'fit: PC2 = {coeffs[0]:.2f} PC1$^2$ {coeffs[1]:+.2f} PC1 {coeffs[2]:+.2f}')
ax.axhline(0, color='0.5', linewidth=0.5)
ax.axvline(0, color='0.5', linewidth=0.5)
ax.set_xlabel('PC1 (ENSO)')
ax.set_ylabel('PC2')
ax.legend(loc='upper left', fontsize=8)
fig.colorbar(sc, ax=ax, label='year');

print(f'correlation of PC2 with PC1: {xr.corr(pc1, pc2).item():.2f}')
print(f'correlation of PC2 with PC1 squared: {xr.corr(pc1**2, pc2).item():.2f}')

## Further reading

- Hannachi, A., Jolliffe, I. T., & Stephenson, D. B. (2007). Empirical orthogonal functions and related techniques in atmospheric science: A review. *International Journal of Climatology*, 27(9), 1119–1152. [doi:10.1002/joc.1499](https://doi.org/10.1002/joc.1499)
- The [`eofs` documentation](https://ajdawson.github.io/eofs/latest/) lists other things the solver can do, e.g. project new data onto the EOFs (`projectField`) or reconstruct the field from a few modes (`reconstructedField`).
- [`xeofs`](https://xeofs.readthedocs.io/) offers rotated EOFs, complex (Hilbert) EOFs, maximum covariance analysis and more.